# ApacheJIT Data Preprocessing v2 — With Feature Engineering

Extends the v1 preprocessing pipeline with derived features and
log-transformation of heavily skewed metrics. This addresses a
limitation identified during Deep MLP baseline evaluation: severe
right-skew (skew > 3 for 9 of 12 features) driven by extreme outlier
commits, which likely constrained model performance under v1
preprocessing.

## 1. Environment Setup

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

## 2. Configuration

In [ ]:
DATA_PATH = "/content/drive/MyDrive/dataset/apachejit_total.csv"
OUTPUT_DIR = "/content/drive/MyDrive/DL-Assignment/data/processed_v2"

RAW_FEATURE_COLUMNS = [
    "la", "ld", "nf", "nd", "ns", "ent",
    "ndev", "age", "nuc", "aexp", "arexp", "asexp",
]
LABEL_COLUMN = "buggy"

# Features with skewness > 1 receive a log1p transform prior to
# scaling. Skewness was measured on the full dataset during EDA;
# 'ent' (skew ~0.93) is left untransformed as its skew is mild.
SKEWED_FEATURES = ["la", "ld", "nf", "nd", "ns", "ndev", "age", "nuc", "aexp", "arexp", "asexp"]

TRAIN_YEAR_MAX = 2014
VAL_YEAR_MIN = 2015
VAL_YEAR_MAX = 2016
TEST_YEAR_MIN = 2017

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 3. Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Year range: {df['year'].min()}-{df['year'].max()}")

## 4. Feature Skewness Diagnosis

Quantifies the skew identified as a limitation in the Deep MLP
baseline results, motivating the transformations applied below.

In [ ]:
skew_before = df[RAW_FEATURE_COLUMNS].skew().sort_values(ascending=False)
print("Skewness (raw features):")
print(skew_before)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flatten(), RAW_FEATURE_COLUMNS):
    ax.hist(df[col], bins=50, color="indianred")
    ax.set_title(f"{col} (skew={df[col].skew():.2f})")
    ax.set_yscale("log")
plt.suptitle("Raw Feature Distributions (Before Log Transform)")
plt.tight_layout()
plt.show()

## 5. Feature Engineering

Two changes are applied:

1. **Derived features** — ratios and totals that may carry signal
   not directly available in the raw metrics (e.g. lines changed per
   file touched, indicating change concentration vs. dispersion).
2. **Log transformation** — `log1p` applied to the 11 features with
   skewness > 1, compressing extreme outlier values (e.g. a 9,967-line
   commit) so they no longer dominate the fitted scaler statistics.

In [ ]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add derived features to the commit metrics dataframe."""
    out = frame.copy()

    out["total_churn"] = out["la"] + out["ld"]
    out["churn_per_file"] = out["total_churn"] / out["nf"].replace(0, 1)
    out["files_per_dir"] = out["nf"] / out["nd"].replace(0, 1)
    out["add_delete_ratio"] = out["la"] / (out["ld"] + 1)  # +1 avoids div-by-zero
    out["exp_ratio"] = out["arexp"] / (out["aexp"] + 1)

    return out


def apply_log_transform(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Apply log1p to specified columns to reduce right-skew."""
    out = frame.copy()
    for col in columns:
        out[col] = np.log1p(out[col])
    return out

In [ ]:
df_engineered = engineer_features(df)

DERIVED_FEATURES = ["total_churn", "churn_per_file", "files_per_dir", "add_delete_ratio", "exp_ratio"]
ALL_FEATURE_COLUMNS = RAW_FEATURE_COLUMNS + DERIVED_FEATURES

print(f"Added {len(DERIVED_FEATURES)} derived features: {DERIVED_FEATURES}")
print(f"Total feature count: {len(ALL_FEATURE_COLUMNS)} (was {len(RAW_FEATURE_COLUMNS)})")

In [ ]:
# Log-transform the skewed raw features and the derived features that
# are themselves skewed (churn-based features inherit skew from la/ld).
LOG_TRANSFORM_COLUMNS = SKEWED_FEATURES + ["total_churn", "churn_per_file"]

df_engineered = apply_log_transform(df_engineered, LOG_TRANSFORM_COLUMNS)

## 6. Verify Skew Reduction

In [ ]:
skew_after = df_engineered[LOG_TRANSFORM_COLUMNS].skew().sort_values(ascending=False)

comparison = pd.DataFrame({
    "skew_before": skew_before.reindex(LOG_TRANSFORM_COLUMNS).fillna(np.nan),
    "skew_after": skew_after,
})
print(comparison)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flatten(), LOG_TRANSFORM_COLUMNS):
    ax.hist(df_engineered[col], bins=50, color="steelblue")
    ax.set_title(f"{col} (skew={df_engineered[col].skew():.2f})")
plt.suptitle("Feature Distributions After Log Transform")
plt.tight_layout()
plt.show()

## 7. Chronological Train / Validation / Test Split

In [ ]:
def chronological_split(frame, train_year_max, val_year_min, val_year_max, test_year_min):
    """Split a dataframe into train/val/test partitions by commit year."""
    train = frame[frame["year"] <= train_year_max].reset_index(drop=True)
    val = frame[
        (frame["year"] >= val_year_min) & (frame["year"] <= val_year_max)
    ].reset_index(drop=True)
    test = frame[frame["year"] >= test_year_min].reset_index(drop=True)
    return train, val, test

In [ ]:
train_df, val_df, test_df = chronological_split(
    df_engineered, TRAIN_YEAR_MAX, VAL_YEAR_MIN, VAL_YEAR_MAX, TEST_YEAR_MIN
)

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} | rows={len(split_df):7d} | buggy_ratio={split_df[LABEL_COLUMN].mean():.4f}")

## 8. Feature and Label Extraction

In [ ]:
def extract_features_labels(frame, feature_cols, label_col):
    X = frame[feature_cols].values.astype(np.float32)
    y = frame[label_col].astype(int).values
    return X, y

In [ ]:
X_train, y_train = extract_features_labels(train_df, ALL_FEATURE_COLUMNS, LABEL_COLUMN)
X_val, y_val = extract_features_labels(val_df, ALL_FEATURE_COLUMNS, LABEL_COLUMN)
X_test, y_test = extract_features_labels(test_df, ALL_FEATURE_COLUMNS, LABEL_COLUMN)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")
print(f"Feature count: {X_train.shape[1]} ({len(RAW_FEATURE_COLUMNS)} raw + {len(DERIVED_FEATURES)} derived)")

## 9. Feature Scaling

As in v1, the scaler is fit exclusively on the training partition.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Post-scaling train mean: {np.round(X_train_scaled.mean(axis=0), 3)}")
print(f"Post-scaling train std:  {np.round(X_train_scaled.std(axis=0), 3)}")

## 10. Class Imbalance Handling

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1]), y=y_train
)
print(f"Class weights [clean, buggy] = {np.round(class_weights, 4)}")

## 11. Persist Processed Artifacts

Saved to a separate `processed_v2` directory so v1 artifacts remain
untouched, allowing a direct before/after comparison between models
trained on v1 (raw, unscaled skew) vs v2 (engineered, log-transformed)
features.

In [ ]:
def save_artifacts(output_dir, arrays, scaler_obj, feature_names):
    os.makedirs(output_dir, exist_ok=True)
    for name, array in arrays.items():
        np.save(os.path.join(output_dir, f"{name}.npy"), array)
    joblib.dump(scaler_obj, os.path.join(output_dir, "scaler.pkl"))
    with open(os.path.join(output_dir, "feature_names.txt"), "w") as f:
        f.write("\n".join(feature_names))

In [ ]:
artifacts = {
    "X_train": X_train_scaled,
    "y_train": y_train,
    "X_val": X_val_scaled,
    "y_val": y_val,
    "X_test": X_test_scaled,
    "y_test": y_test,
    "class_weights": class_weights,
    "class_counts": np.bincount(y_train),  # [n_clean, n_buggy] in the training set,
                                             # so downstream notebooks can build custom
                                             # weight ratios without recomputing from raw data
}

save_artifacts(OUTPUT_DIR, artifacts, scaler, ALL_FEATURE_COLUMNS)
print(f"Artifacts saved to {OUTPUT_DIR}")

## Summary

| Change from v1 | Detail |
|---|---|
| Derived features added | `total_churn`, `churn_per_file`, `files_per_dir`, `add_delete_ratio`, `exp_ratio` |
| Log transform applied | 11 raw features (skew > 1) + 2 derived churn features |
| Feature count | 17 (was 12) |
| Split logic | Unchanged from v1 (chronological, 2003-2014/2015-2016/2017-2019) |
| Class weighting | Unchanged from v1 (`balanced`) |

Output saved to `processed_v2/` — v1 artifacts in `processed/` remain
available for direct comparison.